# FORESEE: DarkPhoton+DarkHiggs

This notebook estimates the sensitivity of forward LHC experiments to the **DarkPhoton+DarkHiggs** model (decay signature). It builds the model from `Models/DarkPhoton+DarkHiggs/build.py`, then walks through the standard FORESEE outputs -- the LLP spectrum at the benchmark point, the production rate versus mass, and the sensitivity reach across detectors -- and can optionally write HepMC event files. Everything model-specific (benchmark point, scan grid, production channels, bounds, plot styling) is parsed from the research notebook `DarkPhoton+DarkHiggs.ipynb` by `routines.get_presets`, so the notebook runs top to bottom without editing.

## 1. Load Libraries

In [ ]:
# Put the library root (the folder containing src/) on the path; this works
# from any notebook location (Examples/ or Models/<Name>/).
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
from matplotlib import pyplot as plt
from src.foresee import Foresee
from src.utils import routines
from src.utils.detectors import default_detectors, DETECTOR_ENERGY

## 2. Specifying the Model

Let us consider a model that consists of a dark photon $A'$ and a dark Higgs $\phi$ as introduced in [2008.12765](https://arxiv.org/abs/2008.12765). As usual, the dark photon $A′$ is a new massive vector boson that kinetically mixes with the SM photon, effectively introducing couplings of the dark photon to all charged SM fermions. The dark Higgs is a new scalar particle that mixes with the SM Higgs, thereby getting Higgs-like couplings to all SM particles. In addition, the dark Higgs generates the dark photon mass: similar to the case f he SM Z-boson, this leads to a coupling of the dark Higgs to a pair of dark photons propoertional to the dark photon mass. The phenomenology of the model can then be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = - \frac{1}{2} \textcolor{red}{m_{A'}}^2 A'^2 - \textcolor{red}{\epsilon} e \sum \bar f \gamma^\mu f A'_\mu - \textcolor{red}{m_{\phi}}^2 \phi^2 -  \sin\textcolor{red}{\theta} \ \sum \ (m \ /\ v )\ \bar f \ f  \ \phi - \textcolor{red}{g}  \textcolor{red}{m_{A'}} \phi A' A'\ . 
\end{equation}

The free parameters of the model are marked in red. 

In the following, we are interested in the regime where $m_{\phi}>2 m_{A'}$ and sufficiently large $g$ such that $BR(\phi \to A' A') \approx 1$. In this case, the dark photon can be produced via $B \to X_s \phi$ with $ \phi \to A' A'$. The corresponding production rate is independent of both $g$, $\epsilon$ and $m_{A'}$, thereby providing an efficient production mechanism throughout the entire parameter space.

The model is built from `Models/DarkPhoton+DarkHiggs/build.py` with the builder's default parameters, and its presets are parsed from `DarkPhoton+DarkHiggs.ipynb` next to it. Change the beam energy or pass further builder overrides through `MODEL_PARAMS` below.

In [ ]:
# Model built from Models/DarkPhoton+DarkHiggs/build.py with the builder's default parameters.
MODEL_NAME = "DarkPhoton+DarkHiggs"

# Beam energy the model is built at, matching DarkPhoton+DarkHiggs.ipynb: "13.6", "14",
# "27", "100", or a fixed-target label like "FT-120GeV".
energy = "14"

# Further overrides forwarded to build_model (see Models/DarkPhoton+DarkHiggs/build.py for
# the accepted parameters), e.g.:
# MODEL_PARAMS = dict(generators_light=["EPOSLHC"])
MODEL_PARAMS = dict()


### Load model and parse presets

`foresee.load_model` builds the model from its `build.py`. `routines.get_presets(MODEL_NAME)` parses the research notebook (without executing it) and returns a plain dict with everything model-specific:

- benchmark point: `mass`, `coupling`
- reach-scan grid: `masses`, `couplings`
- scan setup: `energy`, `setupnames` (one label per production configuration), `modes`
- plot data: `productions`, `branchings`, `bounds`, `bounds2`, `projections`, `lines`, `setups`
- remaining cosmetic plot kwargs: `production_plot`, `reach_plot`

Edit any entry before the cell that consumes it, e.g.

```python
presets["reach_plot"]["title"] = "My custom reach"
presets["masses"] = np.logspace(-2, 0, 20)
```

In [ ]:
foresee = Foresee(path=root + os.sep)
foresee.set_model(model=foresee.load_model(MODEL_NAME, energy=energy, **MODEL_PARAMS))

presets = routines.get_presets(MODEL_NAME)
modelname = presets["modelname"]

print(f"benchmark : mass={presets['mass']} GeV, coupling={presets['coupling']}")
print(f"grid      : {len(presets.get('masses', []))} masses x "
      f"{len(presets.get('couplings', []))} couplings")
print(f"setups    : {presets['setupnames']}")

## 3. Event Generation

In the following, we study the model's benchmark point and export events as a HepMC file.

In [ ]:
mass, coupling = presets["mass"], presets["coupling"]

First, we produce the corresponding flux for this mass and a reference coupling of 1.

In [ ]:
plot = foresee.get_llp_spectrum(mass=mass, coupling=1, do_plot=True)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Spectrum_{modelname}.pdf", bbox_inches="tight")
plot.show()

Next, let us define the detectors. The registry in `src/utils/detectors.py` supplies the standard decay-signature set; we keep the ones running at this beam energy (FASER Run 3 at 13.6 TeV; FASER and FASER2 at the 14 TeV HL-LHC), mirroring the research notebooks' energy-conditional detector list. The first detector is set for the benchmark study below.

In [ ]:
detectors = [d for d in default_detectors("decay") if DETECTOR_ENERGY[d["label"]] == energy]

# Append a custom geometry as set_detector kwargs plus a label, e.g.:
# detectors += [{"label": "FASER_custom", "distance": 480, "length": 1.5,
#                "luminosity": 250, "selection": "np.sqrt(x.x**2 + x.y**2) < 0.1"}]

print("detectors :", [d["label"] for d in detectors])
foresee.set_detector(**{k: v for k, v in detectors[0].items() if k != "label"})

For our benchmark point, let us now look at how many particles decay inside the decay volume. We also export 1000 unweighted events as a HepMC file.

In [ ]:
labels, modes = presets["setupnames"], presets["modes"]

momenta, weights, _ = foresee.write_events(
    mass=mass,
    coupling=coupling,
    energy=energy,
    numberevent=1000,
    filename="model/events/test.hepmc",
    return_data=True,
    weightnames=labels,
    modes=modes,
)

for isetup, setup in enumerate(labels):
    print("Expected number of events for " + setup + ":", round(sum(weights[:, isetup]), 3))

Let us plot the resulting energy distribution.

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = plt.subplot(1,1,1)
energies = [p.e for p in momenta]
for isetup, setup in enumerate(labels):
    ax.hist(energies, weights=weights[:, isetup], bins=np.logspace(2,4, 20+1), histtype='step', label=setup)
ax.set_xscale("log")
ax.set_xlim(1e2,1e4)
ax.set_xlabel("E [GeV]")
ax.set_ylabel("Number of Events per Bin")
ax.legend(frameon=False, labelspacing=0, fontsize=14, loc='upper left')
os.makedirs(f"figures/{modelname}", exist_ok=True)
plt.savefig(f"figures/{modelname}/E_distribution_{modelname}.pdf", bbox_inches="tight")
plt.show()

## 4. Sensitivity Reach

In the following, we obtain the projected sensitivity of the model, on the mass and coupling grid recovered from the research notebook. `routines.cache_spectra` first produces the corresponding fluxes; masses already cached in `model/LLP_spectra/` are skipped. A grid the parser cannot recover (one built from live model calls, as in iDM) falls back to `routines.cached_masses`, i.e. the masses already on disk.

In [ ]:
masses, couplings = presets.get("masses", []), presets["couplings"]

# grids built from live model calls (iDM) are not recoverable by parsing;
# fall back to the masses already cached in model/LLP_spectra/
if len(masses) == 0:
    masses = routines.cached_masses(foresee)

print(f"grid: {len(masses)} masses x {len(couplings)} couplings")
routines.cache_spectra(foresee, masses)

We can now plot the production rate vs mass using `foresee.plot_production`, with the channel groupings, branching-fraction sub-panel, and styling recovered from the research notebook. (The HNL production groupings are built from live model channels, which parsing cannot recover, so this cell is skipped for the HNL models.)

In [ ]:
if presets["productions"]:
    plot = foresee.plot_production(
        masses=masses,
        productions=presets["productions"],
        energy=energy,
        branchings=presets["branchings"],
        **presets["production_plot"],
    )
    if isinstance(plot, tuple):
        plot, ax, ax2 = plot
    os.makedirs(f"figures/{modelname}", exist_ok=True)
    plot.savefig(f"figures/{modelname}/Production_{modelname}.pdf", bbox_inches="tight")
    plot.show()
else:
    print("production groupings not recoverable from the notebook - skipping")

`routines.gen_events` scans the mass x coupling grid for every detector, one result column per label in `setupnames`, and saves each curve to `results/<Name>/<energy>_<detector>_<label>.npy` under the notebook's working directory (pass `outdir` to relocate).

In [ ]:
routines.gen_events(foresee, masses, couplings, labels, modes, detectors)

We can now plot the results with `foresee.plot_reach`, overlaying the existing bounds, projections, and annotated lines from the presets. The `setups` rows come from the research notebook; only the ones whose result file exists are drawn. This run wrote the curves for the current beam energy - re-run the notebook at the other energy (e.g. `energy="13.6"` for FASER Run 3) to add its detectors, exactly as the research notebooks do.

In [ ]:
results_dir = os.path.join("results", modelname)
setups = [[os.path.abspath(os.path.join(results_dir, s[0]))] + list(s[1:])
          for s in presets["setups"] if os.path.exists(os.path.join(results_dir, s[0]))]

# Add rows for any custom detectors scanned above, e.g.:
# setups += [[os.path.abspath(os.path.join(results_dir, f"{energy}TeV_FASER_custom_{labels[0]}.npy")),
#             "FASER custom", "teal", "solid", 0.0, 3]]

plot = foresee.plot_reach(
    setups=setups,
    bounds=presets["bounds"],
    bounds2=presets["bounds2"],
    projections=presets["projections"],
    lines=presets["lines"],
    **presets["reach_plot"],
)
os.makedirs(f"figures/{modelname}", exist_ok=True)
plot.savefig(f"figures/{modelname}/Reach_{modelname}.pdf", bbox_inches="tight")
plot.show()

## 5. Generate HepMC event files

`routines.gen_hepmc` writes one event file per detector x mass x coupling into `results/<Name>/` under the notebook's working directory (pass `outdir` to relocate), here limited to the benchmark point; pass mass/coupling lists to scan a grid.

In [ ]:
routines.gen_hepmc(foresee, [mass], [coupling], labels, modes, detectors, nevent=100)